# Lab 7 :

**Prepared by:** Priyanka Boowade

CSBS, 3rd year https://github.com/PriyankaBoowade/MachineLearning2026.git


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Load the Engineered Data
input_path = Path("../data/processed/olist_orders_feature_engineered.csv")
df = pd.read_csv(input_path)
target = "is_late_delivery"

# Check class balance
print("Target Proportions:\n", df[target].value_counts(normalize=True))

# 2. Define X and y (using numerical features for this baseline)
y = df[target]
X = df.drop(columns=[target])
X = X.select_dtypes(include=np.number)
X = X.fillna(X.median())

# 3. Train-Test Split (Keeping 20% completely unseen)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Train a Baseline Model
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# 5. The Confusion Matrix & Basic Metrics
print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Single Split Metrics ---")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-Score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, zero_division=0))

# 6. K-Fold Cross-Validation (Testing for stability)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring_metrics)

print("\n--- 5-Fold Cross Validation Results ---")
for metric in scoring_metrics:
    values = cv_results["test_" + metric]
    print(f"{metric:10s} mean={values.mean():.3f}  std={values.std():.3f}")

# 7. A More Correct Pipeline-Based Evaluation (Preventing Leakage)
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipeline_cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1")
print("\n--- Pipeline CV F1 Scores ---")
print("Scores:", pipeline_cv_scores)
print("Mean:", pipeline_cv_scores.mean())
print("Std:", pipeline_cv_scores.std())

# 8. Final Unseen Test Set Evaluation
pipeline.fit(X_train, y_train)
final_pred = pipeline.predict(X_test)
print("\n--- Final Pipeline Evaluation on Unseen Test Data ---")
print(classification_report(y_test, final_pred, zero_division=0))

Target Proportions:
 is_late_delivery
0    0.934283
1    0.065717
Name: proportion, dtype: float64


C:\environment\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--- Confusion Matrix ---
[[10471  8111]
 [  383   924]]

--- Single Split Metrics ---
Accuracy: 0.5729297601689376
Precision: 0.10226895406751522
Recall: 0.7069625095638867
F1-Score: 0.17868884161670856
ROC-AUC: 0.6639293836611798

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.96      0.56      0.71     18582
           1       0.10      0.71      0.18      1307

    accuracy                           0.57     19889
   macro avg       0.53      0.64      0.45     19889
weighted avg       0.91      0.57      0.68     19889



C:\environment\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\environment\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer 


--- 5-Fold Cross Validation Results ---
accuracy   mean=0.571  std=0.004
precision  mean=0.100  std=0.002
recall     mean=0.693  std=0.008
f1         mean=0.175  std=0.003
roc_auc    mean=0.654  std=0.005

--- Pipeline CV F1 Scores ---
Scores: [0.17321706 0.17352694 0.17604029 0.17591241 0.1778637 ]
Mean: 0.17531208072064547
Std: 0.0017307396589862258

--- Final Pipeline Evaluation on Unseen Test Data ---
              precision    recall  f1-score   support

           0       0.97      0.56      0.71     18582
           1       0.10      0.72      0.18      1307

    accuracy                           0.57     19889
   macro avg       0.53      0.64      0.44     19889
weighted avg       0.91      0.57      0.67     19889



In [2]:
# The Challenge Exercise: Testing different thresholds
thresholds = [0.30, 0.50, 0.70]

print("Threshold | Precision | Recall | F1")
print("-" * 35)

for t in thresholds:
    # If the probability is greater than the threshold, predict 1 (Late)
    y_pred_t = (y_prob >= t).astype(int)
    
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    
    print(f"   {t:.2f}   |   {p:.3f}   |  {r:.3f} | {f:.3f}")


Threshold | Precision | Recall | F1
-----------------------------------
   0.30   |   0.068   |  0.944 | 0.127
   0.50   |   0.102   |  0.707 | 0.179
   0.70   |   0.140   |  0.043 | 0.066
